In [79]:
import numpy as np
import pennylane as qml
from scipy.linalg import fractional_matrix_power
from scipy.optimize import minimize
from pennylane import numpy as np

In [80]:
m_e  = 1
m_star = 0.16
eps_0 = 1/(4*np.pi)
eps_r = 8.8
eps = eps_r * eps_0
alpha_r_znse = ((m_e*m_star)/eps_r)**2

In [81]:
alpha_dict = {
    "a_1": 35.52322122 * alpha_r_znse,
    "a_2": 6.513143725 * alpha_r_znse,
    "a_3": 1.822142904 * alpha_r_znse,
    "a_4": 0.6259552659 * alpha_r_znse,
    "a_5": 0.2430767471 * alpha_r_znse,
    "a_6": 0.1001124280 * alpha_r_znse
} #sto-6g

In [82]:
N = len(alpha_dict)

In [83]:
def pe_H_func(n, empty_dict, dict_of_alpha):
 
    constant_pe = -m_e**2 / (2 * eps)

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_H_" + str(i) + str(j)
            denominator = (dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)])
            value = 1/denominator
            empty_dict[key] = value * constant_pe
    return empty_dict

In [84]:
def pe_cc_func(n, empty_dict, dict_of_alpha):
 
    constant = -(m_e**2) * ((1/eps_0) - (1/eps))

    r_cc = 3.96739525e-05 #In
    
   

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_cc_" + str(i) + str(j)
            p =  (dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)])
            w1 = 0.85355339
            t1 = 2 - (2**0.5)
            w2 = 0.14644661
            t2 = 2 + (2**0.5)
            w1t1 = (r_cc**2)*w1*np.exp(-p*(r_cc**2)*(t1**2))*t1
            w2t2 = (r_cc**2)*w2*np.exp(-p*(r_cc**2)*(t2**2))*t2
            
            value = w1t1 + w2t2 
            
            empty_dict[key] = value * constant
    return empty_dict

In [85]:
def ke_func(n, empty_dict, dict_of_alpha):
    hbar = 1  
    constant_ke = (3*(hbar**2))/(m_star)

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "T_" + str(i) + str(j)
            numerator = (dict_of_alpha['a_'+str(i)])*(dict_of_alpha['a_'+str(j)])* (np.pi)**1.5
            denominator = ((dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)]))**2.5
            value = numerator/denominator
            empty_dict[key] = value * constant_ke
    return empty_dict

In [86]:
def overlap_func(n, empty_dict, dict_of_alpha):

    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "S_" + str(i) + str(j)
            denominator = ((dict_of_alpha['a_'+str(i)]) + (dict_of_alpha['a_'+str(j)]))
            value = (np.pi/denominator)**1.5
            empty_dict[key] = value
    return empty_dict

In [87]:
def total_pe_construction(n, empty_dict,pe_H_dictionary, pe_cc_dictionary):
    for i in range(1,n+1):
        for j in range(1,n+1):
            key = "V_" + str(i) + str(j)
            value = pe_H_dictionary["V_H_"+str(i)+str(j)]+ pe_cc_dictionary["V_cc_"+str(i)+str(j)]
            empty_dict[key] = value
    return empty_dict

In [88]:
def Hamilt_construction(H,n,ke_dictionary, pe_dictionary):
    H = np.zeros((n,n))
    for i in range(0,n):
        for j in range(0,n):
            H[i][j] = ke_dictionary['T_'+str(i+1)+str(j+1)] + pe_dictionary['V_'+str(i+1)+str(j+1)]
    
    return H

In [89]:
def overlap_matrix_construction(S,n, overlap_dict):
    S = np.zeros((n,n))
    for i in range(0,n):
        for j in range(0,n):
            S[i][j] =  overlap_dict['S_'+str(i+1)+str(j+1)]
    
    return S

In [90]:
V_H_dict = {}
V_H_dict = pe_H_func(N,V_H_dict,alpha_dict)
V_cc_dict = {}
V_cc_dict = pe_cc_func(N,V_cc_dict,alpha_dict)
V_dict = {}
V_dict = total_pe_construction(N,V_dict,V_H_dict,V_cc_dict)
T_dict = {}
T_dict = ke_func(N,T_dict,alpha_dict)
S_dict = {}
S_dict = overlap_func(N,S_dict,alpha_dict)

In [91]:
H = ''
H = Hamilt_construction(H,N,T_dict,V_dict)

In [92]:
S = ''
S = overlap_matrix_construction(S,N,S_dict)

In [93]:
nqubits = 3

In [94]:
def EfficientSU2(theta): # eigen value function
    for i in range(nqubits):
        qml.RY(theta[i], wires = i)
        qml.RZ(theta[i+nqubits], wires = i)
    for i in range(nqubits-1):
        qml.CNOT(wires = [i, i + 1])
    for i in range(nqubits):
        qml.RY(theta[i+(2*nqubits)], wires = i)
        qml.RZ(theta[i+(3*nqubits)], wires = i)


In [95]:
dev = qml.device("default.qubit", wires=nqubits)
@qml.qnode(dev)
def energy(param):
    EfficientSU2(param)
    return qml.expval(H)

In [96]:
dev = qml.device("default.qubit", wires=nqubits)
@qml.qnode(dev)
def energy2(param):
    EfficientSU2(param)
    return qml.expval(S)

In [97]:
def eigen_cal(param):
    h_expec = energy(param)
    s_expec = energy2(param)

    E0 = h_expec/s_expec

    return E0

In [98]:
@qml.qnode(dev)  # eigen vector funtion
def EfficientSU22(theta):
    for i in range(nqubits):
            qml.RY(theta[i], wires = i)
            qml.RZ(theta[i+nqubits], wires = i)
    for i in range(nqubits-1):
        qml.CNOT(wires = [i, i + 1])
    for i in range(nqubits):
        qml.RY(theta[i+(2*nqubits)], wires = i)
        qml.RZ(theta[i+(3*nqubits)], wires = i)
    return qml.state()

In [99]:
pad_size = 8 - H.shape[0]
H = np.pad(H, ((0, pad_size), (0, pad_size)), mode='constant')
H[N][N] =1000
H[N+1][N+1] =1000
S = np.pad(S, ((0, pad_size), (0, pad_size)), mode='constant')
S[N][N] =1
S[N+1][N+1] =1

H = qml.Hermitian(H, wires=list(range(nqubits)))
S = qml.Hermitian(S, wires=list(range(nqubits)))

In [ ]:
theta = np.array(np.random.random(4*nqubits), requires_grad=True)
result = minimize(eigen_cal, x0=theta, tol=10e-8)

In [101]:
result.fun * 27211.38 #eigen_value

-28.06100519011723

In [102]:
result.x # optimised permaters

array([-2.22174607e+00,  5.16519354e-01,  1.20800931e+00, -6.44866461e-04,
       -1.39300557e-04, -3.30811763e-06,  3.08182553e+00, -4.39817681e-01,
       -3.43170223e-01, -7.03681280e-04, -5.52486674e-05,  1.65685768e-06])

In [103]:
E_0_vec = np.abs(EfficientSU22(result.x)) #added optimised permaters to the second circuit to find the eigenvector for E0


In [104]:
E_0_vec 

tensor([0.34683836, 0.23420835, 0.54617146, 0.58370349, 0.38739639,
        0.18625824, 0.02386936, 0.02233244], requires_grad=True)